In [1]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

# Power consumption and Dark Silicon

## How fast are my CPUs and GPUs?

In [2]:
! lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      28
  On-line CPU(s) list:       0-27
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Core(TM) i7-14700K
    CPU family:              6
    Model:                   183
    Thread(s) per core:      2
    Core(s) per socket:      20
    Socket(s):               1
    Stepping:                1
    CPU(s) scaling MHz:      29%
    CPU max MHz:             5600.0000
    CPU min MHz:             800.0000
    BogoMIPS:                6835.20
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush dts acpi mmx fxsr sse 
                             sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm cons
                             tant_tsc art arch_perfmon pebs bts rep_go

In [3]:
! cat /proc/cpuinfo |grep MHz

cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 3211.471
cpu MHz		: 800.000
cpu MHz		: 3852.912
cpu MHz		: 800.000
cpu MHz		: 4364.142
cpu MHz		: 800.000
cpu MHz		: 5500.000
cpu MHz		: 5500.000
cpu MHz		: 5500.000
cpu MHz		: 800.000
cpu MHz		: 4731.781
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 3147.114
cpu MHz		: 4299.995
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 1101.382
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000
cpu MHz		: 800.000


In [4]:
! ssh htseng@eevee "lscpu"

Architecture:                         x86_64
CPU op-mode(s):                       32-bit, 64-bit
Byte Order:                           Little Endian
Address sizes:                        46 bits physical, 48 bits virtual
CPU(s):                               72
On-line CPU(s) list:                  0-71
Thread(s) per core:                   2
Core(s) per socket:                   18
Socket(s):                            2
NUMA node(s):                         2
Vendor ID:                            GenuineIntel
CPU family:                           6
Model:                                85
Model name:                           Intel(R) Xeon(R) Gold 6140 CPU @ 2.30GHz
Stepping:                             4
CPU MHz:                              1266.358
CPU max MHz:                          3700.0000
CPU min MHz:                          1000.0000
BogoMIPS:                             4600.00
Virtualization:                       VT-x
L1d cache:                            1.1 MiB
L1i 

In [5]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power; make -C matrix_mul clean all ;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm 1024 16" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 49742255355 |   0 | 49523239673 | 6.908647e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 17941837225 |   0 | 17801883633 | 2.491922e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 41266250032 |   0 | 40944355464 | 5.731424e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2434 |  28 |          40 |      33.8056 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    992.7915 |   0 |    497.3276 |      13.7888 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |    128.0434 |   0 |     87.2117 |       1.7784 |
|  Runtime (RDTSC) [s] STAT |  1288.1160 |  17.8905 |   17.8905 |  17.8905 |
| Runtime unhalted [s] STAT |     7.8013 |        0 |    7.7402 |   0.1084 |
|      Clock [MHz] STAT     | 48010.7523 | 997.8044 | 1004.9854 | 666.8160 |

In [6]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power; time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm 1024 16" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 49580378776 |   0 | 49523236503 | 6.886164e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 17864399362 |   0 | 17828030988 | 2.481167e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 12151996732 |   0 | 12079645264 | 1.687777e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2591 |  30 |          42 |      35.9861 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    517.7827 |   0 |    281.8334 |       7.1914 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     66.0239 |   0 |     40.5713 |       0.9170 |
|  Runtime (RDTSC) [s] STAT |   380.5488 |   5.2854 |    5.2854 |   5.2854 |
| Runtime unhalted [s] STAT |     7.7672 |        0 |    7.7514 |   0.1079 |
|      Clock [MHz] STAT     | 27725.8294 | 996.6736 | 3394.4555 | 385.0810 |

In [7]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power/popcounts; make clean; make; time sudo likwid-perfctr -g ENERGY ./popcount_D" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 84938159410 |   0 | 80002272871 | 1.179697e+09 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 24571094631 |   0 | 21385587133 | 3.412652e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 56513593664 |   0 | 49186919744 | 7.849110e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2521 |  29 |          41 |      35.0139 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |   1189.2299 |   0 |    595.6612 |      16.5171 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |    153.7303 |   0 |    104.3254 |       2.1351 |
|  Runtime (RDTSC) [s] STAT |  1543.5360 |  21.4380 |   21.4380 |  21.4380 |
| Runtime unhalted [s] STAT |    10.6832 |        0 |    9.2981 |   0.1484 |
|      Clock [MHz] STAT     | 33016.5089 | 998.0612 | 1007.6634 | 458.5626 |

In [8]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power/popcounts;  sudo likwid-perfctr -g ENERGY ./popcount_D" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 80057837660 |   0 | 80002269088 | 1.111914e+09 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 21418394944 |   0 | 21383083820 | 2.974777e+08 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 14572438072 |   0 | 14505431068 | 2.023950e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2681 |  31 |          43 |      37.2361 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    616.8594 |   0 |    333.7104 |       8.5675 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     78.8787 |   0 |     48.4762 |       1.0955 |
|  Runtime (RDTSC) [s] STAT |   454.7304 |   6.3157 |    6.3157 |   6.3157 |
| Runtime unhalted [s] STAT |     9.3129 |        0 |    9.2975 |   0.1293 |
|      Clock [MHz] STAT     | 32705.2416 | 994.7792 | 3390.3290 | 454.2395 |

In [9]:
# Operating at lowest freq.
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/lowerMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm_pthread 1024 16 32" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  |  9598344914 |   0 |  9316611623 | 1.333103e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  |  6491815691 |   0 |  6355446796 | 9.016411e+07 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 14931148096 |   0 | 14617504640 | 2.073771e+08 |
|       TEMP_CORE STAT       |   TMP0  |        2573 |  30 |          42 |      35.7361 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |    358.6567 |   0 |    179.9379 |       4.9813 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |           0 |   0 |           0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |     46.2687 |   0 |     31.7854 |       0.6426 |
|  Runtime (RDTSC) [s] STAT |   462.0312 |   6.4171 |    6.4171 |   6.4171 |
| Runtime unhalted [s] STAT |     2.8227 |        0 |    2.7633 |   0.0392 |
|      Clock [MHz] STAT     | 28987.8129 | 993.3472 | 1004.0248 | 402.6085 |

In [10]:
# Allowing turbo boost
! ssh htseng@eevee "/nfshome/htseng/courses/CS203/demo/power/restoreMaxFreq.sh; cd /nfshome/htseng/courses/CS203/demo/power;  time sudo likwid-perfctr -g ENERGY matrix_mul/blockmm_pthread 1024 16 4" | grep "STAT"

Sleeping longer as likwid_sleep() called without prior initialization
Sleeping longer as likwid_sleep() called without prior initialization
|   INSTR_RETIRED_ANY STAT   |  FIXC0  | 9368227748 |   0 | 9316610499 | 1.301143e+08 |
| CPU_CLK_UNHALTED_CORE STAT |  FIXC1  | 6424591756 |   0 | 6394414571 | 8.923044e+07 |
|  CPU_CLK_UNHALTED_REF STAT |  FIXC2  | 4396309424 |   0 | 4335920532 | 6.105985e+07 |
|       TEMP_CORE STAT       |   TMP0  |       2687 |  31 |         43 |      37.3194 |
|     PWR_PKG_ENERGY STAT    |   PWR0  |   189.4929 |   0 |   103.8143 |       2.6318 |
|     PWR_PP0_ENERGY STAT    |   PWR1  |          0 |   0 |          0 |            0 |
|    PWR_DRAM_ENERGY STAT    |   PWR3  |    24.2475 |   0 |    14.7492 |       0.3368 |
|  Runtime (RDTSC) [s] STAT |   138.0024 |   1.9167 |    1.9167 |   1.9167 |
| Runtime unhalted [s] STAT |     2.7934 |        0 |    2.7802 |   0.0388 |
|      Clock [MHz] STAT     | 18952.3295 | 999.6279 | 3391.9224 | 263.2268 |
|          CP